In [21]:
from langchain.chat_models import init_chat_model
from typing import Annotated,List,Dict,Literal,Optional,Any
import operator
from langchain.agents import  create_agent
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from langchain_community.tools import DuckDuckGoSearchResults
from langchain.agents.middleware import ToolCallLimitMiddleware,ModelRetryMiddleware
import re
from langchain.tools import tool
from langchain.messages import HumanMessage
from langchain_mistralai import  ChatMistralAI
from pydantic import BaseModel,Field
import os
from dotenv import load_dotenv
from langgraph_supervisor import create_supervisor

load_dotenv()

True

In [3]:
model=ChatMistralAI()

In [4]:
model.invoke('hi how are u')

AIMessage(content="Hi! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with anything you need! 😊 How about you—how are you doing today?", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 7, 'total_tokens': 51, 'completion_tokens': 44, 'num_cached_tokens': 0}, 'model_name': 'mistral-small', 'model': 'mistral-small', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019c24f4-d2b4-7cd2-abc4-9f40e3d6b3b7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 44, 'total_tokens': 51})

In [5]:
tool=DuckDuckGoSearchResults(top=100,name='duckduckgo')

In [6]:
class modeloutput(BaseModel):
    model_output: str=Field(..., description="model output")

In [7]:
new_model=model.with_structured_output(modeloutput)

In [11]:
new_model.invoke('how are u')

modeloutput(model_output="I'm doing well, thank you for asking! How can I assist you today?")

In [72]:

llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1-0528",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto", 
    async_client=True,
    huggingfacehub_api_token=os.getenv('HF_TOKEN') ,# let Hugging Face choose the best provider for you,

)


model=ChatHuggingFace(llm=llm)

In [61]:
from langchain.tools import tool
from langchain_core.messages import AIMessage

def filter_response(response: AIMessage) -> str:
    """
    Filters out the reasoning tags from the response of the model.
    """
    content = response.content
    cleaned_content = re.sub(r'<think>.*?</think>\n*', '', content, flags=re.DOTALL).strip()
    return cleaned_content

In [19]:
agent = create_agent(
    model=model,
    tools=[tool],
    name='Researcher-Agent',
    system_prompt='You are a research assistant. Use the search tool to find information and provide detailed analysis.',
    middleware=[ModelRetryMiddleware(max_retries=3)]
      )



In [20]:
await agent.ainvoke({"messages": [HumanMessage("whats the temprature of rajokri delhi.")]}
)

{'messages': [HumanMessage(content='whats the temprature of rajokri delhi.', additional_kwargs={}, response_metadata={}, id='d44378de-c3a9-4983-8a00-db9d5f27cdb6'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'Qt9ahKXsR', 'function': {'name': 'duckduckgo', 'arguments': '{"query": "temperature of rajokri delhi"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 128, 'total_tokens': 148, 'completion_tokens': 20, 'num_cached_tokens': 0}, 'model_name': 'mistral-small', 'model': 'mistral-small', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, name='Researcher-Agent', id='lc_run--019c24f9-650e-70b3-b5cc-60fad8b8d1cb-0', tool_calls=[{'name': 'duckduckgo', 'args': {'query': 'temperature of rajokri delhi'}, 'id': 'Qt9ahKXsR', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 128, 'output_tokens': 20, 'total_tokens': 148}),
  ToolMessage(content='snippet: Get the Rajokri , Delhi , India local hourly forecast incl